# Pretraining Objectives: Causal LM, Masked LM, and Prefix LM

Language models learn by predicting tokens, but **which** tokens they predict -- and **what context** they can see -- defines the pretraining objective. This choice fundamentally shapes what the model is good at.

The three main pretraining objectives are:

| Objective | Context | Predicts | Example Model |
|-----------|---------|----------|---------------|
| **Causal LM** | Left-to-right only | Every next token | GPT, LLaMA |
| **Masked LM** | Full bidirectional | ~15% masked tokens | BERT, RoBERTa |
| **Prefix LM** | Bidirectional on prefix, causal on suffix | Suffix tokens | T5, UL2 |

In this notebook we will implement all three, visualize how they work, compare their loss curves, and explore the tradeoffs between them.

## 1. Imports and Setup

In [ ]:
%matplotlib inline

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from objectives import CausalLMLoss, MaskedLMLoss, PrefixLMLoss, PretrainingObjectiveAnalyzer

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Global parameters
VOCAB_SIZE = 1000
SEQ_LEN = 20
BATCH_SIZE = 4

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Sequence length: {SEQ_LEN}")
print(f"Batch size:      {BATCH_SIZE}")
print(f"Random baseline loss (uniform over vocab): {np.log(VOCAB_SIZE):.4f}")

---
## 2. Causal Language Modeling (GPT-style)

In **Causal LM**, the model sees tokens left-to-right and predicts the next token at every position. Given the sequence `[A, B, C, D]`, the training targets are:

```
Input:   [A, B, C, D]
Target:  [B, C, D, -]   (shifted by 1)
```

This is the objective used by GPT, GPT-2, GPT-3, LLaMA, and most modern generative models.

In [ ]:
causal_loss_fn = CausalLMLoss()
print("CausalLMLoss internals:")
print(f"  Loss function: {causal_loss_fn.loss_fn}")

### 2.1 Input-Target Alignment (Shift by 1)

The key insight: the logits at position `i` are trained to predict the token at position `i+1`. The implementation flattens and shifts:

In [ ]:
# Create a small example to see the shift
example_tokens = torch.tensor([[10, 25, 42, 88, 3, 77, 56, 91]])
print("Input sequence:  ", example_tokens[0].tolist())
print("Target sequence: ", example_tokens[0, 1:].tolist(), " (shifted left by 1)")
print()
print("Position-by-position prediction targets:")
for i in range(len(example_tokens[0]) - 1):
    print(f"  Position {i}: sees tokens[:{i+1}] = {example_tokens[0, :i+1].tolist():<30s} -> predicts token[{i+1}] = {example_tokens[0, i+1].item()}")

In [ ]:
# Compute causal LM loss on random data
torch.manual_seed(42)
input_ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
logits = torch.randn(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)

loss = causal_loss_fn(logits, input_ids)
print(f"Causal LM loss on random data: {loss.item():.4f}")
print(f"Expected (log(vocab_size)):     {np.log(VOCAB_SIZE):.4f}")
print(f"Number of prediction targets:   {BATCH_SIZE * (SEQ_LEN - 1)} (all positions except last, per sequence)")

### 2.2 Why Causal LM Works for Generation

Causal LM is the natural choice for **text generation** because:

1. **Training matches inference**: during generation, the model only sees previous tokens -- exactly the same setup as training.
2. **Every token is a training signal**: unlike Masked LM (only 15% of tokens), causal LM learns from every position.
3. **Autoregressive factorization**: the model implicitly learns `P(x_1) * P(x_2|x_1) * P(x_3|x_1,x_2) * ...`

The downside: the model can only attend to the **left** context, never the right. This limits its usefulness for tasks like classification or NER where bidirectional context helps.

---
## 3. Masked Language Modeling (BERT-style)

In **Masked LM**, we randomly mask ~15% of tokens and train the model to reconstruct them. The model sees the full bidirectional context (both left and right of each masked token).

```
Input:   [A, [MASK], C, [MASK], E]
Target:  [-, B,      -, D,      -]   (loss only on masked positions)
```

This is the objective used by BERT, RoBERTa, and ALBERT.

In [ ]:
masked_loss_fn = MaskedLMLoss(vocab_size=VOCAB_SIZE, mask_token_id=103, mask_prob=0.15)
print("MaskedLMLoss internals:")
print(f"  Vocab size:    {masked_loss_fn.vocab_size}")
print(f"  Mask token ID: {masked_loss_fn.mask_token_id}")
print(f"  Mask prob:     {masked_loss_fn.mask_prob}")

### 3.1 The Masking Process

Let's manually trace through masking to see exactly which tokens get selected:

In [ ]:
torch.manual_seed(42)
input_ids = torch.randint(0, VOCAB_SIZE, (1, SEQ_LEN))

# Simulate the masking process (same logic as MaskedLMLoss.forward)
mask = torch.rand(1, SEQ_LEN) < 0.15
masked_positions = mask[0].nonzero(as_tuple=True)[0].tolist()

print(f"Original tokens:  {input_ids[0].tolist()}")
print(f"Mask positions:   {masked_positions}")
print(f"Masked count:     {mask.sum().item()} / {SEQ_LEN} = {mask.sum().item()/SEQ_LEN:.1%}")
print()

# Show which tokens are masked
display_tokens = []
for i, tok in enumerate(input_ids[0].tolist()):
    if i in masked_positions:
        display_tokens.append(f"[{tok}]")
    else:
        display_tokens.append(str(tok))
print("Visualization (bracketed = masked):")
print("  ", " ".join(display_tokens))

### 3.2 Visualizing Masked Positions

In [ ]:
# Visualize masking on a batch of sequences
torch.manual_seed(42)
batch_ids = torch.randint(0, VOCAB_SIZE, (4, SEQ_LEN))
batch_mask = torch.rand(4, SEQ_LEN) < 0.15

fig, ax = plt.subplots(figsize=(14, 3))

# Create color grid: green=visible, red=masked
colors = np.zeros((4, SEQ_LEN, 3))
for i in range(4):
    for j in range(SEQ_LEN):
        if batch_mask[i, j]:
            colors[i, j] = [0.9, 0.3, 0.3]  # red for masked
        else:
            colors[i, j] = [0.3, 0.8, 0.4]  # green for visible

ax.imshow(colors, aspect='auto')

# Add token IDs as text
for i in range(4):
    for j in range(SEQ_LEN):
        tok = batch_ids[i, j].item()
        label = "[M]" if batch_mask[i, j] else str(tok)
        fontsize = 6 if not batch_mask[i, j] else 7
        ax.text(j, i, label, ha='center', va='center', fontsize=fontsize,
                fontweight='bold' if batch_mask[i, j] else 'normal', color='white')

ax.set_xlabel('Position')
ax.set_ylabel('Batch')
ax.set_title('Masked LM: Token Visibility (green=visible, red=masked)')
ax.set_xticks(range(SEQ_LEN))
ax.set_yticks(range(4))
ax.set_yticklabels([f'Seq {i}' for i in range(4)])
plt.tight_layout()
plt.show()

total_masked = batch_mask.sum().item()
print(f"Total masked tokens: {total_masked} / {4 * SEQ_LEN} = {total_masked/(4*SEQ_LEN):.1%}")

In [ ]:
# Compute masked LM loss on random data
torch.manual_seed(42)
input_ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
logits = torch.randn(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)

loss = masked_loss_fn(logits, input_ids)
print(f"Masked LM loss on random data: {loss.item():.4f}")
print(f"Expected (log(vocab_size)):     {np.log(VOCAB_SIZE):.4f}")
print(f"Approx prediction targets:      {int(BATCH_SIZE * SEQ_LEN * 0.15)} (only ~15% of all tokens)")

---
## 4. Prefix Language Modeling (T5-style)

**Prefix LM** is a hybrid: the model sees the entire prefix with bidirectional attention, then autoregressively predicts the suffix.

```
Sequence:  [A, B, C, D, E, F, G, H]
                       ^--- prefix_len (50%)
Prefix:    [A, B, C, D]            <- full bidirectional attention
Suffix:              [E, F, G, H]  <- causal (next-token prediction)
Loss:                 on suffix only
```

This combines the benefits of both worlds: bidirectional understanding of the prefix, plus generative capability on the suffix.

In [ ]:
prefix_loss_fn = PrefixLMLoss(prefix_ratio=0.5)
print("PrefixLMLoss internals:")
print(f"  Prefix ratio: {prefix_loss_fn.prefix_ratio}")
print(f"  For seq_len={SEQ_LEN}: prefix_len={int(SEQ_LEN * 0.5)}, suffix_len={SEQ_LEN - int(SEQ_LEN * 0.5)}")

### 4.1 Prefix/Suffix Split Visualization

In [ ]:
torch.manual_seed(42)
example_tokens = torch.randint(0, VOCAB_SIZE, (1, SEQ_LEN))
prefix_len = int(SEQ_LEN * 0.5)

print(f"Full sequence ({SEQ_LEN} tokens):")
print(f"  {example_tokens[0].tolist()}")
print()
print(f"Prefix ({prefix_len} tokens) -- bidirectional attention, no loss:")
print(f"  {example_tokens[0, :prefix_len].tolist()}")
print()
print(f"Suffix ({SEQ_LEN - prefix_len} tokens) -- causal attention, loss computed here:")
print(f"  {example_tokens[0, prefix_len:].tolist()}")
print()

# Visualize the split
fig, ax = plt.subplots(figsize=(14, 1.5))

for i in range(SEQ_LEN):
    color = '#4ECDC4' if i < prefix_len else '#FF6B6B'
    ax.add_patch(plt.Rectangle((i, 0), 0.9, 0.9, facecolor=color, edgecolor='white', linewidth=2))
    ax.text(i + 0.45, 0.45, str(example_tokens[0, i].item()), ha='center', va='center',
            fontsize=7, fontweight='bold', color='white')

ax.set_xlim(-0.2, SEQ_LEN + 0.2)
ax.set_ylim(-0.3, 1.3)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Prefix LM: Prefix (teal) vs Suffix (red) -- loss computed on red only', fontsize=11)

prefix_patch = mpatches.Patch(color='#4ECDC4', label=f'Prefix ({prefix_len} tokens, bidirectional)')
suffix_patch = mpatches.Patch(color='#FF6B6B', label=f'Suffix ({SEQ_LEN - prefix_len} tokens, causal + loss)')
ax.legend(handles=[prefix_patch, suffix_patch], loc='lower center', ncol=2, fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Compute prefix LM loss on random data
torch.manual_seed(42)
input_ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
logits = torch.randn(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)

loss = prefix_loss_fn(logits, input_ids)
suffix_len = SEQ_LEN - int(SEQ_LEN * 0.5)
print(f"Prefix LM loss on random data: {loss.item():.4f}")
print(f"Expected (log(vocab_size)):     {np.log(VOCAB_SIZE):.4f}")
print(f"Number of prediction targets:   {BATCH_SIZE * (suffix_len - 1)} (suffix positions minus 1 for shift)")

---
## 5. Side-by-Side Comparison of All Three Objectives

Let's visualize what each objective "sees" and "predicts" on the same input sequence:

In [ ]:
torch.manual_seed(42)
seq = torch.randint(0, VOCAB_SIZE, (1, SEQ_LEN))
prefix_len = int(SEQ_LEN * 0.5)

# Determine masked positions (for MLM visualization)
torch.manual_seed(42)
mlm_mask = torch.rand(1, SEQ_LEN) < 0.15
masked_positions = set(mlm_mask[0].nonzero(as_tuple=True)[0].tolist())

fig, axes = plt.subplots(3, 1, figsize=(14, 5))

token_list = seq[0].tolist()

# --- Causal LM ---
for i in range(SEQ_LEN):
    if i < SEQ_LEN - 1:
        color = '#2196F3'  # blue -- prediction target
    else:
        color = '#BBBBBB'  # grey -- no target (last position)
    axes[0].add_patch(plt.Rectangle((i, 0), 0.9, 0.9, facecolor=color, edgecolor='white', linewidth=2))
    axes[0].text(i + 0.45, 0.45, str(token_list[i]), ha='center', va='center',
                 fontsize=7, fontweight='bold', color='white')
axes[0].set_title(f'Causal LM: predict every next token ({SEQ_LEN - 1} targets)', fontsize=10)
axes[0].set_xlim(-0.2, SEQ_LEN + 0.2)
axes[0].set_ylim(-0.1, 1.1)
axes[0].axis('off')

# --- Masked LM ---
for i in range(SEQ_LEN):
    if i in masked_positions:
        color = '#F44336'  # red -- masked (prediction target)
    else:
        color = '#4CAF50'  # green -- visible (no loss)
    axes[1].add_patch(plt.Rectangle((i, 0), 0.9, 0.9, facecolor=color, edgecolor='white', linewidth=2))
    label = "[M]" if i in masked_positions else str(token_list[i])
    axes[1].text(i + 0.45, 0.45, label, ha='center', va='center',
                 fontsize=7, fontweight='bold', color='white')
axes[1].set_title(f'Masked LM: predict masked tokens only ({len(masked_positions)} targets)', fontsize=10)
axes[1].set_xlim(-0.2, SEQ_LEN + 0.2)
axes[1].set_ylim(-0.1, 1.1)
axes[1].axis('off')

# --- Prefix LM ---
for i in range(SEQ_LEN):
    if i < prefix_len:
        color = '#4ECDC4'  # teal -- prefix (no loss)
    else:
        color = '#FF6B6B'  # red -- suffix (prediction target)
    axes[2].add_patch(plt.Rectangle((i, 0), 0.9, 0.9, facecolor=color, edgecolor='white', linewidth=2))
    axes[2].text(i + 0.45, 0.45, str(token_list[i]), ha='center', va='center',
                 fontsize=7, fontweight='bold', color='white')
axes[2].set_title(f'Prefix LM: predict suffix tokens ({SEQ_LEN - prefix_len} in suffix, {SEQ_LEN - prefix_len - 1} loss targets after shift)', fontsize=10)
axes[2].set_xlim(-0.2, SEQ_LEN + 0.2)
axes[2].set_ylim(-0.1, 1.1)
axes[2].axis('off')

plt.suptitle('Pretraining Objectives: What Gets Predicted?', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Training Simulation: Loss Curves

Let's use the `PretrainingObjectiveAnalyzer` to simulate training steps with random data and compare how the loss behaves for each objective:

In [ ]:
analyzer = PretrainingObjectiveAnalyzer(vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN)
print(f"Analyzer created:")
print(f"  vocab_size: {analyzer.vocab_size}")
print(f"  seq_len:    {analyzer.seq_len}")

In [ ]:
# Simulate 100 training steps
torch.manual_seed(42)
losses = analyzer.simulate_training(num_steps=100)

print("Loss statistics over 100 steps:")
for name, vals in losses.items():
    clean_vals = [v for v in vals if v is not None]
    if clean_vals:
        print(f"  {name:<12s}: mean={np.mean(clean_vals):.4f}, std={np.std(clean_vals):.4f}, min={np.min(clean_vals):.4f}, max={np.max(clean_vals):.4f}")

In [ ]:
# Plot loss curves using the built-in method
analyzer.plot_loss_curves(losses)

**Observations:**

Since we are using random logits (no actual learning), all three losses hover around `log(vocab_size)`. In real training you would see all three decrease, but at different rates. The fluctuation you see is from the stochastic masking in Masked LM and the smaller number of prediction targets in Prefix LM.

---
## 7. Masking Rate Ablation

For Masked LM, the masking probability is a critical hyperparameter. BERT uses 15%, but what happens with different rates? Let's test: 0%, 5%, 15%, 30%, and 50%.

In [ ]:
torch.manual_seed(42)
mask_results = analyzer.analyze_masking_strategies()

print("\nMasking strategy results:")
for strategy, loss_val in mask_results.items():
    print(f"  {strategy:<20s}: avg loss = {loss_val:.4f}")

### 7.1 Loss vs Mask Probability (Custom Visualization)

In [ ]:
# Recompute with more granularity for a smooth curve
torch.manual_seed(42)
mask_probs = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
avg_losses = []
std_losses = []

for mp in mask_probs:
    loss_fn = MaskedLMLoss(VOCAB_SIZE, mask_prob=mp)
    step_losses = []
    for _ in range(50):
        batch = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
        logits = torch.randn(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)
        l = loss_fn(logits, batch).item()
        step_losses.append(l)
    avg_losses.append(np.mean(step_losses))
    std_losses.append(np.std(step_losses))

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(mask_probs, avg_losses, yerr=std_losses, marker='o', capsize=5,
            linewidth=2, markersize=8, color='steelblue')
ax.axvline(x=0.15, color='red', linestyle='--', alpha=0.7, label='BERT default (15%)')
ax.axhline(y=np.log(VOCAB_SIZE), color='gray', linestyle=':', alpha=0.5, label=f'Random baseline (ln({VOCAB_SIZE}))')
ax.set_xlabel('Mask Probability', fontsize=12)
ax.set_ylabel('Average Loss', fontsize=12)
ax.set_title('Masked LM: Impact of Masking Rate on Loss', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Note: With random logits, all masking rates produce similar loss (~ln(vocab)).")
print("In real training, the masking rate affects the tradeoff between:")
print("  - Low rate: few training signals per sequence (slow learning)")
print("  - High rate: too much corruption (harder to learn meaningful patterns)")

---
## 8. Manual Head-to-Head: Same Input, All Three Losses

Let's compute all three losses on the exact same input and logits to directly compare:

In [ ]:
torch.manual_seed(42)

# Same input for all
input_ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
logits = torch.randn(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)

# Compute all three
causal_l = causal_loss_fn(logits, input_ids)
masked_l = masked_loss_fn(logits, input_ids)
prefix_l = prefix_loss_fn(logits, input_ids)

print("Losses on identical input (batch_size={}, seq_len={}, vocab_size={}):".format(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE))
print(f"  Causal LM:  {causal_l.item():.4f}")
print(f"  Masked LM:  {masked_l.item():.4f}")
print(f"  Prefix LM:  {prefix_l.item():.4f}")
print(f"  Random baseline: {np.log(VOCAB_SIZE):.4f}")

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 5))

names = ['Causal LM', 'Masked LM', 'Prefix LM']
values = [causal_l.item(), masked_l.item(), prefix_l.item()]
colors = ['#2196F3', '#F44336', '#FF9800']

bars = ax.bar(names, values, color=colors, alpha=0.8, edgecolor='white', linewidth=2)
ax.axhline(y=np.log(VOCAB_SIZE), color='gray', linestyle='--', alpha=0.6, label=f'Random baseline ln({VOCAB_SIZE}) = {np.log(VOCAB_SIZE):.2f}')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Head-to-Head: Loss on Same Input', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 9. Number of Prediction Targets per Objective

A key difference between the objectives is how many tokens contribute to the loss. More targets per sequence means more learning signal per forward pass.

In [ ]:
# Compute exact target counts
torch.manual_seed(42)
mlm_mask = torch.rand(BATCH_SIZE, SEQ_LEN) < 0.15
prefix_len = int(SEQ_LEN * 0.5)

targets = {
    'Causal LM': BATCH_SIZE * (SEQ_LEN - 1),
    'Masked LM': int(mlm_mask.sum().item()),
    'Prefix LM': BATCH_SIZE * (SEQ_LEN - prefix_len - 1),
}

total_tokens = BATCH_SIZE * SEQ_LEN

print(f"Total tokens in batch: {total_tokens}")
print()
for name, count in targets.items():
    pct = count / total_tokens * 100
    print(f"  {name:<12s}: {count:3d} targets ({pct:5.1f}% of all tokens)")

In [ ]:
# Visualize target counts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of absolute counts
names = list(targets.keys())
counts = list(targets.values())
colors = ['#2196F3', '#F44336', '#FF9800']

bars = axes[0].bar(names, counts, color=colors, alpha=0.8, edgecolor='white', linewidth=2)
axes[0].axhline(y=total_tokens, color='gray', linestyle='--', alpha=0.5, label=f'Total tokens ({total_tokens})')
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(count), ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Number of Prediction Targets', fontsize=11)
axes[0].set_title('Prediction Targets per Objective', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Pie chart of signal efficiency
percentages = [c / total_tokens * 100 for c in counts]
axes[1].bar(names, percentages, color=colors, alpha=0.8, edgecolor='white', linewidth=2)
for i, (n, p) in enumerate(zip(names, percentages)):
    axes[1].text(i, p + 1, f'{p:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[1].set_ylabel('% of Tokens Used as Targets', fontsize=11)
axes[1].set_title('Training Signal Efficiency', fontsize=12)
axes[1].set_ylim(0, 110)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Causal LM uses almost every token as a training signal (most efficient per sequence).")
print("Masked LM uses only ~15%, meaning it needs more data to get the same number of gradient updates.")
print("Prefix LM falls in between, using ~50% of tokens (the suffix).")

---
## 10. Attention Mask Patterns

The three objectives differ fundamentally in what each token is allowed to attend to. Let's visualize the attention mask patterns:

In [ ]:
n = 8  # small sequence for clarity

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Causal LM: lower-triangular mask
causal_mask = torch.tril(torch.ones(n, n))
axes[0].imshow(causal_mask, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Causal LM\n(lower triangular)', fontsize=12)
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')
for i in range(n):
    for j in range(n):
        axes[0].text(j, i, int(causal_mask[i, j].item()), ha='center', va='center',
                     fontsize=9, color='white' if causal_mask[i, j] > 0.5 else 'black')

# Masked LM: full attention (all ones)
full_mask = torch.ones(n, n)
axes[1].imshow(full_mask, cmap='Greens', vmin=0, vmax=1)
axes[1].set_title('Masked LM\n(full bidirectional)', fontsize=12)
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
for i in range(n):
    for j in range(n):
        axes[1].text(j, i, 1, ha='center', va='center', fontsize=9, color='white')

# Prefix LM: full attention for prefix, causal for suffix
prefix_n = n // 2
prefix_mask = torch.zeros(n, n)
# Prefix tokens can see all prefix tokens (bidirectional within prefix)
prefix_mask[:prefix_n, :prefix_n] = 1
# Suffix tokens can see all prefix tokens + causal within suffix
for i in range(prefix_n, n):
    prefix_mask[i, :prefix_n] = 1  # can see all prefix
    prefix_mask[i, prefix_n:i+1] = 1  # causal within suffix

axes[2].imshow(prefix_mask, cmap='Oranges', vmin=0, vmax=1)
axes[2].set_title('Prefix LM\n(bidir prefix + causal suffix)', fontsize=12)
axes[2].set_xlabel('Key position')
axes[2].set_ylabel('Query position')
for i in range(n):
    for j in range(n):
        axes[2].text(j, i, int(prefix_mask[i, j].item()), ha='center', va='center',
                     fontsize=9, color='white' if prefix_mask[i, j] > 0.5 else 'black')

# Add prefix/suffix boundary lines to prefix LM plot
axes[2].axhline(y=prefix_n - 0.5, color='red', linewidth=2, linestyle='--')
axes[2].axvline(x=prefix_n - 0.5, color='red', linewidth=2, linestyle='--')

plt.suptitle('Attention Mask Patterns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Gradient Signal Analysis

Let's measure how many gradient updates each objective provides per batch by looking at which positions contribute to the loss:

In [ ]:
torch.manual_seed(42)

# Run multiple batches and track statistics
num_trials = 100
causal_targets_per_batch = []
masked_targets_per_batch = []
prefix_targets_per_batch = []

for _ in range(num_trials):
    # Causal: always (seq_len - 1) * batch_size targets
    causal_targets_per_batch.append(BATCH_SIZE * (SEQ_LEN - 1))

    # Masked: stochastic, depends on random mask
    mask = torch.rand(BATCH_SIZE, SEQ_LEN) < 0.15
    masked_targets_per_batch.append(mask.sum().item())

    # Prefix: always (suffix_len - 1) * batch_size targets
    suffix_len = SEQ_LEN - int(SEQ_LEN * 0.5)
    prefix_targets_per_batch.append(BATCH_SIZE * (suffix_len - 1))

print(f"Gradient targets per batch (averaged over {num_trials} batches):")
print(f"  Causal LM:  {np.mean(causal_targets_per_batch):.1f} +/- {np.std(causal_targets_per_batch):.1f}  (deterministic)")
print(f"  Masked LM:  {np.mean(masked_targets_per_batch):.1f} +/- {np.std(masked_targets_per_batch):.1f}  (stochastic)")
print(f"  Prefix LM:  {np.mean(prefix_targets_per_batch):.1f} +/- {np.std(prefix_targets_per_batch):.1f}  (deterministic)")
print()
print("Causal LM provides {:.1f}x more gradient signal than Masked LM per batch.".format(
    np.mean(causal_targets_per_batch) / np.mean(masked_targets_per_batch)))

In [ ]:
# Distribution of masked targets (the only stochastic one)
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(masked_targets_per_batch, bins=range(int(min(masked_targets_per_batch)) - 1,
        int(max(masked_targets_per_batch)) + 3), alpha=0.7, color='#F44336',
        edgecolor='white', linewidth=1.5, label='Masked LM')

ax.axvline(x=np.mean(causal_targets_per_batch), color='#2196F3', linewidth=3,
           linestyle='-', label=f'Causal LM (fixed = {int(np.mean(causal_targets_per_batch))})')
ax.axvline(x=np.mean(prefix_targets_per_batch), color='#FF9800', linewidth=3,
           linestyle='-', label=f'Prefix LM (fixed = {int(np.mean(prefix_targets_per_batch))})')
ax.axvline(x=np.mean(masked_targets_per_batch), color='#F44336', linewidth=2,
           linestyle='--', label=f'Masked LM mean = {np.mean(masked_targets_per_batch):.1f}')

ax.set_xlabel('Number of Prediction Targets per Batch', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Training Targets per Batch', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 12. Multiple Runs: Loss Variability

Let's run multiple independent evaluations to understand the variability of each loss:

In [ ]:
torch.manual_seed(42)

n_eval = 200
causal_losses = []
masked_losses = []
prefix_losses = []

for _ in range(n_eval):
    ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
    lg = torch.randn(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE)

    causal_losses.append(causal_loss_fn(lg, ids).item())
    masked_losses.append(masked_loss_fn(lg, ids).item())
    prefix_losses.append(prefix_loss_fn(lg, ids).item())

fig, ax = plt.subplots(figsize=(10, 5))

bp = ax.boxplot([causal_losses, masked_losses, prefix_losses],
                labels=['Causal LM', 'Masked LM', 'Prefix LM'],
                patch_artist=True, widths=0.5)

colors = ['#2196F3', '#F44336', '#FF9800']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.axhline(y=np.log(VOCAB_SIZE), color='gray', linestyle='--', alpha=0.5,
           label=f'ln({VOCAB_SIZE}) = {np.log(VOCAB_SIZE):.2f}')
ax.set_ylabel('Loss', fontsize=12)
ax.set_title(f'Loss Distribution ({n_eval} evaluations on random data)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Masked LM shows higher variance because the random mask changes each time.")
print("Fewer prediction targets = noisier loss estimate per batch.")

---
## 13. Key Insights and Tradeoffs

### When to Use Each Objective

| Objective | Best For | Strengths | Weaknesses |
|-----------|----------|-----------|------------|
| **Causal LM** | Text generation, code generation, dialogue | Natural for generation; high training signal efficiency (all tokens) | Unidirectional -- cannot leverage right context |
| **Masked LM** | Classification, NER, QA, sentence embeddings | Bidirectional context; strong representations | Only 15% of tokens are training signal; train-test mismatch ([MASK] tokens) |
| **Prefix LM** | Conditional generation, translation, summarization | Bidirectional understanding of input + generative output | More complex; fewer training signals than causal LM |

### Training Signal Efficiency

- **Causal LM**: ~95% of tokens are targets (all except the last in each sequence)
- **Prefix LM**: ~45% of tokens are targets (suffix - 1 for the shift)
- **Masked LM**: ~15% of tokens are targets (with standard masking rate)

This means Causal LM extracts roughly **6x more learning signal** per batch than Masked LM, which partially explains why modern large language models predominantly use the causal objective.

### The Bidirectionality Tradeoff

- Masked LM can see **both directions**, making its representations richer for understanding tasks.
- Causal LM sees only the **left context**, but this aligns perfectly with generation.
- Prefix LM strikes a **middle ground**: bidirectional on the input (prefix), generative on the output (suffix).

### Modern Trends

Most recent large models (GPT-4, LLaMA, Claude, Gemini) use **Causal LM** because:
1. It scales efficiently with model size and data.
2. It provides maximum training signal per token.
3. Instruction tuning can recover strong understanding capabilities despite the unidirectional limitation.

Encoder-only models (BERT-style) with Masked LM remain dominant for **embedding** and **classification** tasks where bidirectional context is critical.

In [ ]:
# Summary table
print("=" * 70)
print("PRETRAINING OBJECTIVES SUMMARY")
print("=" * 70)
print(f"{'Objective':<14s} {'Context':<22s} {'Targets':<20s} {'Models'}")
print("-" * 70)
print(f"{'Causal LM':<14s} {'Left-to-right only':<22s} {'All tokens (~95%)':<20s} {'GPT, LLaMA'}")
print(f"{'Masked LM':<14s} {'Full bidirectional':<22s} {'~15% masked tokens':<20s} {'BERT, RoBERTa'}")
print(f"{'Prefix LM':<14s} {'Bidir prefix + causal':<22s} {'Suffix (~45%)':<20s} {'T5, UL2'}")
print("=" * 70)